In [ ]:
!pip install ultralytics aim

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    device = torch.device("cpu")
    print("GPU is not available. Using CPU instead.")

In [ ]:
import os
import random
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import aim
import torch
import yaml
from ultralytics import YOLO

DATA_ROOT = Path("data/frames").resolve()
IMAGES_DIR = DATA_ROOT / "images"
LABELS_DIR = DATA_ROOT / "labels"
CLASSES_FILE = DATA_ROOT / "classes.txt"
TRAIN_LIST = DATA_ROOT / "train.txt"
VAL_LIST = DATA_ROOT / "val.txt"
DATA_YAML = DATA_ROOT / "data.yaml"

MODEL = "yolo26m.pt"
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 8
WORKERS = min(8, os.cpu_count() or 1)
VAL_FRACTION = 0.20
SEED = 42
PATIENCE = 30
DEVICE = 0 if torch.cuda.is_available() else "cpu"

AIM_EXPERIMENT = "yolo26m-abandoned-objects"
RUN_NAME = datetime.now(timezone.utc).strftime("yolo26m-%Y%m%d-%H%M%S")

print(f"Dataset: {DATA_ROOT}")
print(f"Device:  {DEVICE}")

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg"}


def read_classes() -> list[str]:
    if not CLASSES_FILE.is_file():
        raise FileNotFoundError(f"Missing {CLASSES_FILE}")
    names = [line.strip() for line in CLASSES_FILE.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
    if not names:
        raise ValueError(f"{CLASSES_FILE} contains no class names")
    return names


def validate_label(path: Path, class_count: int) -> Counter:
    counts = Counter()
    if not path.exists():  # A missing label is a valid background image in YOLO.
        return counts
    for line_number, line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"{path}:{line_number}: expected 5 values, found {len(parts)}")
        raw_class, *raw_box = parts
        class_value = float(raw_class)
        class_id = int(class_value)
        if class_value != class_id or not 0 <= class_id < class_count:
            raise ValueError(f"{path}:{line_number}: invalid class id {raw_class}")
        cx, cy, width, height = map(float, raw_box)
        if not all(0.0 <= value <= 1.0 for value in (cx, cy, width, height)):
            raise ValueError(f"{path}:{line_number}: box coordinates must be in [0, 1]")
        if width <= 0.0 or height <= 0.0:
            raise ValueError(f"{path}:{line_number}: width and height must be positive")
        counts[class_id] += 1
    return counts


def prepare_dataset() -> tuple[list[str], dict[str, list[Path]], dict[str, Counter]]:
    class_names = read_classes()
    if not IMAGES_DIR.is_dir():
        raise FileNotFoundError(f"Missing {IMAGES_DIR}")
    if not LABELS_DIR.is_dir():
        raise FileNotFoundError(f"Missing {LABELS_DIR}")

    images = sorted(path for path in IMAGES_DIR.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)
    if len(images) < 2:
        raise ValueError(f"Need at least 2 images in {IMAGES_DIR}; found {len(images)}")

    labels_by_image = {image: LABELS_DIR / f"{image.stem}.txt" for image in images}
    all_counts = Counter()
    missing_labels = []
    for image, label in labels_by_image.items():
        if not label.exists():
            missing_labels.append(image.name)
        all_counts.update(validate_label(label, len(class_names)))

    shuffled = images.copy()
    random.Random(SEED).shuffle(shuffled)
    val_size = max(1, min(len(shuffled) - 1, round(len(shuffled) * VAL_FRACTION)))
    splits = {"val": sorted(shuffled[:val_size]), "train": sorted(shuffled[val_size:])}

    split_counts = {}
    for split, split_images in splits.items():
        counts = Counter()
        for image in split_images:
            label = labels_by_image[image]
            counts.update(validate_label(label, len(class_names)))
        split_counts[split] = counts

    TRAIN_LIST.write_text("\n".join(str(path) for path in splits["train"]) + "\n", encoding="utf-8")
    VAL_LIST.write_text("\n".join(str(path) for path in splits["val"]) + "\n", encoding="utf-8")

    config = {
        "path": str(DATA_ROOT),
        "train": TRAIN_LIST.name,
        "val": VAL_LIST.name,
        "names": {index: name for index, name in enumerate(class_names)},
    }
    DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")

    print(f"Classes: {dict(enumerate(class_names))}")
    print(f"Images:  {len(images)} total, {len(splits['train'])} train, {len(splits['val'])} validation")
    print(f"Boxes:   {dict(all_counts)}")
    if missing_labels:
        print(f"Note: {len(missing_labels)} image(s) have no label file and will be treated as background.")
    for split, counts in split_counts.items():
        named_counts = {class_names[index]: counts[index] for index in range(len(class_names))}
        print(f"{split:5s}: {len(splits[split]):5d} images, boxes {named_counts}")
    print(f"Dataset YAML: {DATA_YAML}")
    return class_names, splits, split_counts


class_names, splits, split_counts = prepare_dataset()

## 4. Create the Aim run and metric callback

Ultralytics provides per-class precision, recall, and F1 at its selected max-F1 confidence operating point, plus AP50. This notebook tracks their class means after every validation epoch using the same Aim naming convention as `data_prep.ipynb`: `Precision`, `Recall`, `F1`, and `mAP@0.5`.

In [ ]:
run = aim.Run(experiment=AIM_EXPERIMENT)
run.name = RUN_NAME
run["hyperparameters"] = {
    "model": MODEL,
    "epochs": EPOCHS,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "workers": WORKERS,
    "patience": PATIENCE,
    "seed": SEED,
    "device": str(DEVICE),
    "validation_fraction": VAL_FRACTION,
}
run["dataset"] = {
    "source": str(DATA_ROOT),
    "yaml": str(DATA_YAML),
    "classes": class_names,
    "train_images": len(splits["train"]),
    "validation_images": len(splits["val"]),
}


def log_validation_metrics_to_aim(trainer) -> None:
    # on_fit_epoch_end is called after validation, when DetMetrics is populated.
    if getattr(trainer, "rank", -1) not in (-1, 0):
        return
    metrics = trainer.metrics
    required = ("metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)")
    if not metrics or any(key not in metrics for key in required):
        print(f"No validation metrics available at epoch {trainer.epoch}")
        return

    precision = float(metrics["metrics/precision(B)"])
    recall = float(metrics["metrics/recall(B)"])
    map50 = float(metrics["metrics/mAP50(B)"])
    box_metrics = trainer.validator.metrics.box
    f1 = float(box_metrics.f1.mean()) if len(box_metrics.f1) else 0.0
    epoch = int(trainer.epoch)

    values = {"Precision": precision, "Recall": recall, "F1": f1, "mAP@0.5": map50}
    for name, value in values.items():
        run.track(value, name=name, epoch=epoch)
    print("Aim | " + ", ".join(f"{name}={value:.4f}" for name, value in values.items()))


print(f"Aim run: {run.hash} ({AIM_EXPERIMENT} / {RUN_NAME})")

### Train

In [ ]:
model = YOLO(MODEL)
model.add_callback("on_fit_epoch_end", log_validation_metrics_to_aim)

try:
    train_results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        workers=WORKERS,
        device=DEVICE,
        patience=PATIENCE,
        seed=SEED,
        deterministic=True,
        pretrained=True,
        amp=True,
        val=True,
        plots=True,
    )
    best_weights = Path(model.trainer.best)
    last_weights = Path(model.trainer.last)
    run["artifacts"] = {"best_weights": str(best_weights), "last_weights": str(last_weights)}
finally:
    run.close()

print(f"Best weights: {best_weights}")
print(f"Last weights: {last_weights}")

In [ ]:
best_model = YOLO(str(best_weights))
best_metrics = best_model.val(data=str(DATA_YAML), imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device=DEVICE)
best_precision = float(best_metrics.box.mp)
best_recall = float(best_metrics.box.mr)
best_f1 = float(best_metrics.box.f1.mean()) if len(best_metrics.box.f1) else 0.0
print({
    "precision": best_precision,
    "recall": best_recall,
    "f1": best_f1,
    "map50": float(best_metrics.box.map50),
})